# Pinwheel: separability of a three-class dataset

Each class is a stick, the segment from $(a,1)$ to $(b,1)$ rotated by $0^\circ$, $120^\circ$, $240^\circ$.
For the current $(a,b)$ we report whether the hulls are pairwise disjoint, which classes split off from the other two,
and whether the dataset is separable: a centered $W\ne0$ with $y_i\ge y_j$ for every $x$ in class $i$, $j\ne i$.
When it is, the regions $R_i=\{y_i=\max y\}$ of a separating $W$ are shaded.

Separability is decided by the LP: centered $W$, all margins $(y_i-y_j)(x)\ge0$, margins summing to $1$
(valid since the dataset is full rank).

Requires `numpy`, `scipy`, `matplotlib`, `ipywidgets`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from matplotlib.colors import ListedColormap
from scipy.optimize import linprog

COLORS = ["red", "green", "blue"]
LIM = 5.0


def rot(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s], [s, c]])


def sticks(a, b):
    """Endpoints of the three sticks; the hull of each class is its stick."""
    base = np.array([[a, 1.0], [b, 1.0]])
    return [base @ rot(2 * np.pi * i / 3).T for i in range(3)]


def cross(u, v):
    return u[0] * v[1] - u[1] * v[0]


def segments_meet(p, q, r, s):
    d = lambda o, x, y: cross(x - o, y - o)
    return d(p, q, r) * d(p, q, s) <= 0 and d(r, s, p) * d(r, s, q) <= 0


def pairwise_disjoint(C):
    return all(not segments_meet(*C[i], *C[j]) for i in range(3) for j in range(i + 1, 3))


def lift(P):
    return np.hstack([P, np.ones((len(P), 1))])


def splits_off(C, m):
    """Is there (w,b) != 0 with class m on {>= 0} and the other classes on {<= 0}?"""
    Z = np.vstack([lift(C[m])] + [-lift(C[i]) for i in range(3) if i != m])
    # Z u >= 0, sum(Z u) = 1
    res = linprog(np.zeros(3), A_ub=-Z, b_ub=np.zeros(len(Z)),
                  A_eq=Z.sum(0, keepdims=True), b_eq=[1.0],
                  bounds=[(None, None)] * 3, method="highs")
    return res.status == 0


def separating_weight(C):
    """Return a centered separating W (3 x 3, rows (w_i, b_i)) or None."""
    rows, nvar = [], 9
    for i in range(3):
        for z in lift(C[i]):
            for j in range(3):
                if j == i:
                    continue
                g = np.zeros(nvar)
                g[3 * i:3 * i + 3] += z
                g[3 * j:3 * j + 3] -= z
                rows.append(g)            # margin (y_i - y_j)(x) = g . vec(W)
    G = np.array(rows)
    A_eq = np.vstack([G.sum(0)] + [np.tile(np.eye(3)[k], 3) for k in range(3)])  # normalization, centering
    b_eq = np.array([1.0, 0.0, 0.0, 0.0])
    res = linprog(np.zeros(nvar), A_ub=-G, b_ub=np.zeros(len(G)),
                  A_eq=A_eq, b_eq=b_eq, bounds=[(None, None)] * nvar, method="highs")
    return res.x.reshape(3, 3) if res.status == 0 else None

In [2]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, FloatSlider

COLORS = ["red", "green", "blue"]
LIM = 5.0
g = np.linspace(-LIM, LIM, 400)
X1, X2 = np.meshgrid(g, g)
GRID = np.stack([X1.ravel(), X2.ravel(), np.ones(X1.size)], axis=1)
CMAP = ListedColormap(["#ffd6d6", "#d6f5d6", "#d6dcff"])


def show(a=-4.0, b=1.7):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal")
    if b <= a:
        ax.set_title("need a < b"); plt.show(); return
    C = sticks(a, b)
    disjoint = pairwise_disjoint(C)
    ovr = [m for m in range(3) if splits_off(C, m)]
    W = separating_weight(C)
    if W is not None:
        R = np.argmax(GRID @ W.T, axis=1).reshape(X1.shape)
        ax.imshow(R, extent=(-LIM, LIM, -LIM, LIM), origin="lower", cmap=CMAP,
                  vmin=0, vmax=2, alpha=0.9, interpolation="nearest")
        verdict = f"SEPARABLE ({'one vs rest' if ovr else 'wedges'}); shaded: regions $R_i$ of a separating $W$"
    else:
        verdict = "NOT SEPARABLE"
    for P, c in zip(C, COLORS):
        ax.plot(P[:, 0], P[:, 1], color=c, lw=4, solid_capstyle="round")
    ax.plot(0, 0, "k+", ms=8)
    names = ", ".join(COLORS[m] for m in ovr) or "none"
    ax.set_title(f"hulls pairwise disjoint: {disjoint}    splits off: {names}\n{verdict}",
                 fontsize=10, loc="left")
    plt.show()


interact(show,
         a=FloatSlider(value=-4.0, min=-5.0, max=0.0, step=0.05, continuous_update=False),
         b=FloatSlider(value=1.7, min=-1.0, max=2.0, step=0.05, continuous_update=False));

interactive(children=(FloatSlider(value=-4.0, continuous_update=False, description='a', max=0.0, min=-5.0, ste…

Try $(a,b)=(-2,0.3)$ (one vs rest), $(-2,1.5)$ (wedges, nothing splits off), $(-4,1.7)$ (not separable, hulls still pairwise disjoint).